# 第 4 周练习解答 —— USSDforge（USSD 代码生成 / 转换 / 校验）

## 练习目标

用 **Anthropic Claude** + **Gradio** 做一个面向非洲电信场景的小工具 **USSDforge**：

- **生成**：根据业务描述，产出符合尼日利亚电信习惯（MTN / Airtel / Glo）的 USSD 会话流代码
- **转换**：在 Python / Node.js / C++ 之间互转 USSD 代码
- **校验**：让模型只做语法（syntax）检查，不做风格/逻辑点评

## 和本课第 4 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 多模态之外的「代码助手」 | `client.messages.create` + 严格 system prompt |
| Gradio 交互界面 | `gr.Blocks` + Tabs（Generate / Convert / Validate） |
| Prompt 约束输出格式 | 要求「只返回原始代码、不要 markdown 围栏」 |

## 怎么跑

1. 准备环境变量 `ANTHROPIC_API_KEY`
2. 安装依赖：`anthropic`、`gradio`
3. 从上到下运行单元格，最后 `app.launch(...)` 打开界面


In [ ]:
# ========== 安装依赖：Anthropic SDK + Gradio UI ==========
# ! 开头是 notebook shell 魔法：在当前内核环境里 pip 安装包

# anthropic：调用 Claude Messages API；gradio：搭浏览器界面
!pip install anthropic gradio


In [ ]:
# ========== 导入、读 API Key、初始化 Claude 客户端与常量 ==========

# os：读环境变量（Environment Variables）
import os
# anthropic：官方 SDK，后面用 client.messages.create
import anthropic
# gradio：Web UI（本格先导入，后面建界面会用到）
import gradio as gr
# ast / subprocess / tempfile：本笔记本后续若做本地语法检查可用（当前核心路径主要走模型校验）
import ast
import subprocess
import tempfile

# 从环境变量取 Anthropic API Key；没有则后面立刻报错，避免静默失败
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

# 缺密钥就抛 EnvironmentError；错误文案保持英文原样（依赖判断/提示用）
if not ANTHROPIC_API_KEY:
    raise EnvironmentError(
        "ANTHROPIC_API_KEY not found. Set it as an environment variable before running."
    )

# 用密钥创建 Anthropic 客户端
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
# 模型 id：必须与 Anthropic 可用模型名一致，勿擅自改写
MODEL = "claude-opus-4-6"
# 单次回复最大 token 上限（生成/转换时用）
MAX_TOKENS = 8096


In [ ]:
# ========== 核心业务：生成 / 转换 / 语法校验 三个函数 ==========

# 下拉框可选的目标语言列表（UI 与转换逻辑共用）
SUPPORTED_LANGUAGES = ["Python", "Node.js", "C++"]

# System Prompt：角色 + 尼日利亚电信 USSD 规范 +「只输出裸代码」硬约束
# 注意：prompt 字符串必须保持英文原样，改动会改变模型行为
SYSTEM_PROMPT = """You are an expert USSD developer specializing in African telco systems.
You generate clean, production-ready USSD session flow code following Nigerian telco standards (MTN, Airtel, Glo).

Rules:
- Use CON to continue the session and END to terminate it
- Always include input validation
- Structure code with clear session state handling
- Add brief comments only where necessary
- For C++ code, always include ALL necessary headers at the top (e.g. #include <iostream>, #include <string>, #include <vector>, #include <map>, #include <sstream>) before writing any code
- You MUST return raw code only
- Do NOT wrap code in markdown fences or backticks
- Do NOT include any explanation, preamble or summary before or after the code
- The very first character of your response must be the first character of the code"""


def generate_ussd_code(business_description, language):
    """根据业务描述 + 目标语言，调用 Claude 生成 USSD 应用代码。"""
    # user prompt：把业务描述与语言塞进模板；同样保持英文原样
    prompt = f"""Generate a USSD application in {language} for this business:

{business_description}

Return ONLY the raw {language} code. No markdown fences, no explanations, no preamble."""

    # Messages API：system 定规则，user 放具体生成任务
    response = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": prompt}]
    )
    # content[0].text：取第一条文本块；strip 去掉首尾空白
    return response.content[0].text.strip()


def convert_ussd_code(source_code, source_language, target_language):
    """把已有 USSD 代码从 source_language 转到 target_language。"""
    # 转换任务的 user prompt（英文原样保留）
    prompt = f"""Convert this USSD code from {source_language} to {target_language}.

{source_code}

Return ONLY the raw converted {target_language} code. No markdown fences, no explanations."""

    # 同样走 Messages API；system 仍用 SYSTEM_PROMPT 约束「只返回裸代码」
    response = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip()


def validate_syntax(code, language):
    """让模型只做语法检查：通过则固定成功句，失败则列行号与简述。"""
    # 校验专用 user prompt（英文原样保留）
    prompt = f"""You are a strict syntax validator. Analyze this {language} code for syntax errors only.

{code}

Rules:
- Check for syntax errors only, not logic or style issues
- If syntax is correct, respond with exactly: "Syntax check passed. No errors found."
- If there are errors, list each error with its line number and a brief description
- Do not suggest fixes, do not explain the code"""

    # 校验可输出更长；system 换成「严格语法检查器」短设定
    response = client.messages.create(
        model=MODEL,
        max_tokens=16000,
        system="You are a strict code syntax validator. Be precise and concise.",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip()


In [ ]:
# ========== Gradio 界面：Generate / Convert / Validate 三个 Tab ==========

# 前端 JS：把文本写入剪贴板；给「Copy」按钮当 js= 回调用（不走 Python fn）
copy_js = """
async (text) => {
    await navigator.clipboard.writeText(text);
    return text;
}
"""

def build_ui():
    """组装 Blocks 应用：三个 Tab 分别绑定上面三个业务函数。"""
    # title 会出现在浏览器标签页；Blocks 是多组件布局容器
    with gr.Blocks(title="USSDforge") as app:
        # 页头说明（UI 文案字符串保持原样）
        gr.Markdown("# USSDforge\nGenerate and convert USSD flow code for African telco systems.")

        # Tabs：生成 / 转换 / 校验 三个工作区
        with gr.Tabs():

            # ----- Tab 1：根据业务描述生成代码 -----
            with gr.Tab("Generate"):
                # 业务描述多行输入
                description_input = gr.Textbox(
                    label="Business Description",
                    placeholder="e.g. A savings app with balance check, deposit, and withdrawal menu",
                    lines=5
                )
                # 目标语言下拉；choices 来自 SUPPORTED_LANGUAGES
                language_select = gr.Dropdown(
                    label="Output Language",
                    choices=SUPPORTED_LANGUAGES,
                    value="Python"
                )
                # 并排：生成按钮 + 复制按钮
                with gr.Row():
                    generate_btn = gr.Button("Generate Code", variant="primary")
                    copy_gen_btn = gr.Button("Copy Code", variant="secondary")
                # 只读输出框展示生成结果
                generate_output = gr.Textbox(label="Generated USSD Code", lines=25, interactive=False)
                # 点击 → 调 generate_ussd_code(描述, 语言) → 写入输出框
                generate_btn.click(fn=generate_ussd_code, inputs=[description_input, language_select], outputs=generate_output)
                # fn=None + js=copy_js：纯前端复制，不跑 Python
                copy_gen_btn.click(fn=None, inputs=generate_output, outputs=generate_output, js=copy_js)

            # ----- Tab 2：源语言 → 目标语言转换 -----
            with gr.Tab("Convert"):
                # 粘贴待转换源码
                source_code_input = gr.Textbox(label="Source Code", lines=20)
                # From / To 两个语言下拉
                with gr.Row():
                    source_lang_select = gr.Dropdown(label="From", choices=SUPPORTED_LANGUAGES, value="Python")
                    target_lang_select = gr.Dropdown(label="To", choices=SUPPORTED_LANGUAGES, value="Node.js")
                with gr.Row():
                    convert_btn = gr.Button("Convert Code", variant="primary")
                    copy_conv_btn = gr.Button("Copy Code", variant="secondary")
                convert_output = gr.Textbox(label="Converted Code", lines=25, interactive=False)
                # 绑定 convert_ussd_code(源码, 源语言, 目标语言)
                convert_btn.click(fn=convert_ussd_code, inputs=[source_code_input, source_lang_select, target_lang_select], outputs=convert_output)
                copy_conv_btn.click(fn=None, inputs=convert_output, outputs=convert_output, js=copy_js)

            # ----- Tab 3：粘贴代码做语法校验 -----
            with gr.Tab("Validate"):
                validate_code_input = gr.Textbox(label="Paste Code to Validate", lines=20)
                validate_lang_select = gr.Dropdown(label="Language", choices=SUPPORTED_LANGUAGES, value="Python")
                with gr.Row():
                    validate_btn = gr.Button("Check Syntax", variant="primary")
                    copy_val_btn = gr.Button("Copy Result", variant="secondary")
                validate_output = gr.Textbox(label="Validation Result", lines=4, interactive=False)
                # 绑定 validate_syntax(代码, 语言)
                validate_btn.click(fn=validate_syntax, inputs=[validate_code_input, validate_lang_select], outputs=validate_output)
                copy_val_btn.click(fn=None, inputs=validate_output, outputs=validate_output, js=copy_js)

    # 返回组装好的 Blocks，供 launch 使用
    return app


In [ ]:
# ========== 启动 Gradio 应用 ==========

# 调用上一格的 build_ui()，得到 Blocks 实例
app = build_ui()
# launch：拉起本地 Web 服务
# share=True → 尝试生成公网临时链接；inbrowser=True → 尽量自动打开浏览器
app.launch(
   
    share=True,
    inbrowser=True
   
)
